# Model Predict Fraud in the Future
Using the is_fraud column that we created before, we will build a model to predict transactions that may be fraud in the future. In this case, since we already have a column that determines whether or not something is fraud, we will use a supervised model. In particular, we will use RandomForest since we will be able to determine the features that are more important in predicting fraud. In addition to this, we are looking to find non-linear relationships that wouldn't be able to be seen in other models.

In [3]:
import pandas as pd
import numpy as np
import sys
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from pyspark.sql import functions as F, Window
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

In [4]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 21:09:43 WARN Utils: Your hostname, CompuPau, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/20 21:09:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/20 21:09:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
transactions = spark.read.parquet("../data/curated/transactions_with_is_fraud")
transactions = transactions.withColumn("order_datetime", F.to_date("order_datetime"))

print(f"Rows: {transactions.count():,}")
transactions.printSchema()

Rows: 13,614,675
root
 |-- order_id: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- consumer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- fraud_probability: double (nullable = true)
 |-- is_same_day_duplicate: boolean (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- category: string (nullable = true)
 |-- revenue_band: string (nullable = true)
 |-- take_rate: double (nullable = true)
 |-- category_group: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- n_transactions: long (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- avg_merchant_fraud_prob: double (nullab

In [6]:
# see range of date
date_bounds = transactions.select(
    F.min("order_datetime").alias("min_date"),
    F.max("order_datetime").alias("max_date")
).first()

print(f"Date range: {date_bounds['min_date']} -> {date_bounds['max_date']}")

Date range: 2021-02-28 -> 2022-10-26


# Feature Selection

**Categorical variables**
`gender`, `state`, `category_group`, `revenue_band`: we will use these features to observe whether the gender and location of a consumer can influence the decision to commit fraud. In addition to this, we add ' category_group ' and revenue_band to see if the type of items sold can determine whether there is fraud or not.

**Numerical variables**
`dollar_value`, `take_rate`: these features will allow us to determine if a transaction is fraud based on the amount of money spent by the consumer and the average order value that a merchant has. We will also consider the take rate to see whether having a higher take rate can lead to fraud. 


In [7]:
# horizon: we test the model to predict 5 months ahead
CUTOFF_DATE = pd.Timestamp("2022-06-01")

In [8]:
# take data of merchant before cutoff
pre_cutoff = transactions.filter(F.col("order_datetime") < CUTOFF_DATE)
merchant_agg = (pre_cutoff
    .filter(F.col("dollar_value") > 0)
    .groupBy("merchant_abn")
    .agg(
        F.sum("dollar_value").alias("total_revenue_train"),
        F.count("dollar_value").alias("n_transactions_train"),
        F.avg("dollar_value").alias("avg_order_value_train"),
    ))

merchant_agg_pdf = merchant_agg.toPandas()
print(f"Merchants pre-cutoff: {len(merchant_agg_pdf):,}")

/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Merchants pre-cutoff: 4,025


**Add more features**
- `days_since_last_txn`: how many days have past since the last purchase.
  `is_first_txn`: first purchase of the consumer
- `day_of_week`: day of the week, see if it's more likely to have fraud in a certain day
- `dollar_vs_merchant_avg_ratio`: see how big the purchase is compared to the average sell in that place.

In [9]:
user_window = Window.partitionBy("user_id").orderBy("order_datetime") \
    .rowsBetween(Window.unboundedPreceding, -1)
user_order_window = Window.partitionBy("user_id").orderBy("order_datetime")

txn_with_user_feats = (transactions
    .withColumn("user_avg_dollar_so_far", F.avg("dollar_value").over(user_window))
    .withColumn("user_n_txn_so_far", F.count("dollar_value").over(user_window))
    .withColumn(
        "dollar_ratio_to_user_avg",
        F.col("dollar_value") / F.coalesce(F.col("user_avg_dollar_so_far"), F.col("dollar_value"))
    )
    # days since the last purchase of the consumer
    .withColumn(
        "days_since_last_txn",
        F.datediff(F.col("order_datetime"), F.lag("order_datetime").over(user_order_window))
    )
    .withColumn("day_of_week", F.dayofweek("order_datetime"))
)

In [10]:
train_path = "../data/curated/tmp_model_features_train"
test_path = "../data/curated/tmp_model_features_test"

feature_select_cols = [
    "user_id", "merchant_abn", "order_datetime", "dollar_value", "gender", "state",
    "category_group", "revenue_band", "take_rate", "is_fraud",
    "user_avg_dollar_so_far", "user_n_txn_so_far", "dollar_ratio_to_user_avg",
    "days_since_last_txn", "day_of_week",
]

(txn_with_user_feats
    .filter(F.col("order_datetime") < CUTOFF_DATE)
    .select(*feature_select_cols)
    .write.mode("overwrite").parquet(train_path))

(txn_with_user_feats
    .filter(F.col("order_datetime") >= CUTOFF_DATE)
    .select(*feature_select_cols)
    .write.mode("overwrite").parquet(test_path))

print(train_path, "and", test_path)

../data/curated/tmp_model_features_train and ../data/curated/tmp_model_features_test


In [11]:
def load_split(path, merchant_agg_pdf):
    df = pd.read_parquet(path)
    df["order_datetime"] = pd.to_datetime(df["order_datetime"])
    df = df.merge(merchant_agg_pdf, on="merchant_abn", how="left")

    df["log_dollar_value"] = np.log1p(df["dollar_value"])
    df["revenue_per_txn_train"] = df["total_revenue_train"] / df["n_transactions_train"]
    df["log_revenue_per_txn_train"] = np.log1p(df["revenue_per_txn_train"])
    df["log_dollar_ratio_to_user_avg"] = np.log1p(df["dollar_ratio_to_user_avg"].fillna(0))
    df["user_n_txn_so_far"] = df["user_n_txn_so_far"].fillna(0)


    df["dollar_vs_merchant_avg_ratio"] = df["dollar_value"] / df["avg_order_value_train"]
    df["log_dollar_vs_merchant_avg_ratio"] = np.log1p(df["dollar_vs_merchant_avg_ratio"].fillna(0))

    df["is_first_txn"] = df["days_since_last_txn"].isna().astype(int)
    df["days_since_last_txn"] = df["days_since_last_txn"].fillna(-1)

    # cast variables
    for c in ["gender", "state", "category_group", "revenue_band", "merchant_abn"]:
        df[c] = df[c].astype("category")
    df["day_of_week"] = df["day_of_week"].astype("category")

    float_cols = [
        "dollar_value", "take_rate", "avg_order_value_train", "log_dollar_value",
        "log_revenue_per_txn_train", "log_dollar_ratio_to_user_avg", "user_n_txn_so_far",
        "log_dollar_vs_merchant_avg_ratio", "days_since_last_txn",
    ]
    for c in float_cols:
        df[c] = df[c].astype("float32")

    return df

train = load_split(train_path, merchant_agg_pdf)
print(f"Train: {len(train):,} rows ({train['is_fraud'].mean():.1%} fraud) "
      f"| till {train['order_datetime'].max().date()}")
print(f"  memory: {train.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

test = load_split(test_path, merchant_agg_pdf)
print(f"Test:  {len(test):,} rows ({test['is_fraud'].mean():.1%} fraud) "
      f"| from {test['order_datetime'].min().date()}")
print(f"  memory: {test.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

Train: 9,847,495 rows (9.0% fraud) | till 2022-05-31
  memory: 1,256 MB
Test:  3,767,180 rows (9.0% fraud) | from 2022-06-01
  memory: 481 MB


In [12]:
for name, df in [("train", train), ("test", test)]:
    n_null = df["is_fraud"].isna().sum()
    print(f"{name}: {n_null:,} rows  with fraud label {len(df):,} "
          f"({n_null / len(df):.1%}) -> Nan")

# train and test set only use values that have a label
train = train[train["is_fraud"].notna()].copy()
test = test[test["is_fraud"].notna()].copy()

train["is_fraud"] = train["is_fraud"].astype(int)
test["is_fraud"] = test["is_fraud"].astype(int)

print(f"\nTrain final: {len(train):,} rows ({train['is_fraud'].mean():.1%} fraud)")
print(f"Test final:  {len(test):,} rows ({test['is_fraud'].mean():.1%} fraud)")

train: 5,763,784 rows  with fraud label 9,847,495 (58.5%) -> Nan
test: 2,209,723 rows  with fraud label 3,767,180 (58.7%) -> Nan

Train final: 4,083,711 rows (9.0% fraud)
Test final:  1,557,457 rows (9.0% fraud)


In [13]:
# columns that are going to be used for model
num_cols = [
    "log_dollar_value",
    "log_dollar_ratio_to_user_avg",
    "user_n_txn_so_far",
    "days_since_last_txn",
    "log_dollar_vs_merchant_avg_ratio",  
]
cat_cols = ["gender", "state", "category_group", "revenue_band", "day_of_week", "is_first_txn"]
feature_cols = num_cols + cat_cols

Check features behaviour with LogisticsRegression.

In [14]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=200, class_weight="balanced")),
])

In [15]:
baseline.fit(train[feature_cols], train["is_fraud"])
proba_base = baseline.predict_proba(test[feature_cols])[:, 1]

print(f"Baseline (Logistics Regression)")
print(f"  ROC-AUC:  {roc_auc_score(test['is_fraud'], proba_base):.3f}")
print(f"  PR-AUC:   {average_precision_score(test['is_fraud'], proba_base):.3f}  "
      f"(compare against rate of fraud = {test['is_fraud'].mean():.3f})")

Baseline (Logistics Regression)
  ROC-AUC:  0.589
  PR-AUC:   0.122  (compare against rate of fraud = 0.090)


In [16]:
# use balance, since most of the values are not fraud
model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=100,     
        max_depth=15,         
        min_samples_leaf=20,   
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
])

In [17]:
# fit the model with is_fraud column
model.fit(train[feature_cols], train["is_fraud"])
# predict model
y_proba = model.predict_proba(test[feature_cols])[:, 1]
print(f"AUC (split): {roc_auc_score(test['is_fraud'], y_proba):.3f}")
print(f"Random Forest — split")
print(f"  ROC-AUC: {roc_auc_score(test['is_fraud'], y_proba):.3f}")
print(f"  PR-AUC:  {average_precision_score(test['is_fraud'], y_proba):.3f}  "
      f"(compare against rate of fraud = {test['is_fraud'].mean():.3f})")

AUC (split): 0.676
Random Forest — split
  ROC-AUC: 0.676
  PR-AUC:  0.191  (compare against rate of fraud = 0.090)


In [18]:
y_pred = (y_proba >= 0.5).astype(int)
print(classification_report(test["is_fraud"], y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.938     0.718     0.813   1417098
           1      0.154     0.519     0.238    140359

    accuracy                          0.700   1557457
   macro avg      0.546     0.619     0.525   1557457
weighted avg      0.867     0.700     0.761   1557457



In [19]:
ohe = model.named_steps["prep"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(cat_feature_names)
importances = model.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": all_feature_names, "importance": importances}).sort_values("importance", ascending=False)
print(imp_df.head(15))

                               feature  importance
0                     log_dollar_value    0.136545
1         log_dollar_ratio_to_user_avg    0.090119
4     log_dollar_vs_merchant_avg_ratio    0.088789
33                      revenue_band_c    0.083824
32                      revenue_band_b    0.080724
22  category_group_gifts_hobby_novelty    0.068037
20       category_group_furniture_home    0.061072
31                      revenue_band_a    0.057569
29     category_group_tv_media_telecom    0.035155
23        category_group_health_beauty    0.035139
17  category_group_books_media_digital    0.035027
25                category_group_music    0.031974
27       category_group_outdoor_supply    0.027501
19             category_group_footwear    0.026964
16   category_group_antiques_art_craft    0.022616


In [20]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(train, groups=train["merchant_abn"]))

train_g = train.iloc[train_idx]
test_g = train.iloc[test_idx]

overlap = set(train_g["merchant_abn"]) & set(test_g["merchant_abn"])
print(f"Common merchants between train and test: {len(overlap)}")

m_group = clone(model)
m_group.fit(train_g[feature_cols], train_g["is_fraud"])
proba_group = m_group.predict_proba(test_g[feature_cols])[:, 1]
print(f"AUC (merchants that were never explored nunca during training): "
      f"{roc_auc_score(test_g['is_fraud'], proba_group):.3f}")

Common merchants between train and test: 0
AUC (merchants that were never explored nunca during training): 0.473


The model has high scores overall; however, this is mainly because the model is only learning from the features of avg_order_value_train, log_revenue_per_txn_train and take_rate. Thus, instead of predicting a transaction by a learnt pattern from the is_fraud column, it seems that the model only learns what happens with the behaviour of the numerical features for individual merchants.

# Summary of results
From `01_Label_merchants.ipynb`, we know that the fraud probability of merchants is mainly due to the `total_revenue` and `n_transactions`. In particular, a transaction will have a lower probability of being fraud when the transaction being made gives more revenue to the merchant, or when there is a higher number of transactions made.

From `02_Label_consumer.ipynb`, we know that `dollar_value` is what affects the probability of having fraud, specifically when this value is higher than $571. Therefore, an item that is more expensive will increase the probability of a consumer committing fraud.

Lastly, with this model we were able to see that `dollar_value` is the one that has more weight in predicting if a transaction will be fraud. However, it is important to acknowledge that the model is not able to generalise well for the merchants that it didn't know about. In addition to this, it was not possible to find more insights about the is_fraud column, even though we included more features.